# Diemtigen XAI pipeline: 3-segment features + EBM only

This single notebook replaces the previous multi-notebook workflow.
It:
1. builds the **3-segment signal features** dataset,
2. trains/evaluates an **Explainable Boosting Machine (EBM)**,
3. provides native EBM interpretation plots and summaries.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

from interpret.glassbox import ExplainableBoostingClassifier


In [ ]:
data_root = Path('diemtigen_data')
h5_path = data_root / 'events_mainshocks_foreshocks_aftershocks_15sec_23days.h5'
info_path = data_root / 'info_h5_events_mainshocks_foreshocks_aftershocks_15sec_23days.csv'
features_path = data_root / 'data_features.csv'

info = pd.read_csv(info_path)
label_map = {'foreshock': 0, 'aftershock': 1}
info_fa = info[info['event_type'].isin(label_map)].copy().reset_index(drop=True)
info_fa['label'] = info_fa['event_type'].map(label_map)

info_fa[['event_id', 'event_type', 'label']].head()


In [ ]:
def rms(x):
    x = np.asarray(x, dtype=float)
    return float(np.sqrt(np.mean(x * x))) if x.size else np.nan

def energy(x):
    x = np.asarray(x, dtype=float)
    return float(np.sum(x * x)) if x.size else np.nan

def peak_abs(x):
    x = np.asarray(x, dtype=float)
    return float(np.max(np.abs(x))) if x.size else np.nan

def zero_crossing_rate(x):
    x = np.asarray(x, dtype=float)
    if x.size < 2:
        return np.nan
    s = np.sign(x)
    s[s == 0] = 1
    return float(np.mean(s[:-1] != s[1:]))

def safe_ratio(a, b):
    if b is None or np.isnan(b) or abs(b) < 1e-12:
        return np.nan
    return float(a / b)


In [ ]:
SEGMENTS = {
    'noise': (0.0, 5.0),
    'p': (5.0, 8.0),
    'coda': (8.0, 15.0),
}

CHANNELS = {'E': 'HHE', 'N': 'HHN', 'Z': 'HHZ'}


def get_event_traces(h5f, event_id: str):
    g = h5f[str(event_id)]
    tr = g['traces']
    keys = list(tr.keys())
    sr = float(tr[keys[0]].attrs.get('sampling_rate', np.nan))

    out = {}
    for comp, chan in CHANNELS.items():
        k = [kk for kk in keys if kk.endswith(chan)]
        out[comp] = np.array(tr[k[0]][:]) if k else None
    return out, sr


In [ ]:
rows = []

with h5py.File(h5_path, 'r') as h5f:
    for _, r in info_fa.iterrows():
        eid = str(r['event_id'])
        if eid not in h5f:
            continue

        traces, sr = get_event_traces(h5f, eid)
        if any(traces[c] is None for c in CHANNELS):
            continue

        L = min(traces['E'].size, traces['N'].size, traces['Z'].size)
        for c in traces:
            traces[c] = traces[c][:L].astype(float)

        base = {
            'event_id': eid,
            'label': int(r['label']),
            'event_type': r['event_type'],
            'sampling_rate': sr,
            'npts': L,
        }

        feat = dict(base)
        for seg_name, (t0, t1) in SEGMENTS.items():
            i0, i1 = int(round(t0 * sr)), int(round(t1 * sr))
            for comp in ['E', 'N', 'Z']:
                x = traces[comp][i0:i1]
                prefix = f'{comp}_{seg_name}'
                feat[f'{prefix}_rms'] = rms(x)
                feat[f'{prefix}_energy'] = energy(x)
                feat[f'{prefix}_peak_abs'] = peak_abs(x)
                feat[f'{prefix}_zcr'] = zero_crossing_rate(x)

        for comp in ['E', 'N', 'Z']:
            feat[f'{comp}_snr_proxy'] = safe_ratio(feat[f'{comp}_p_rms'], feat[f'{comp}_noise_rms'])
            feat[f'{comp}_coda_to_p_energy'] = safe_ratio(feat[f'{comp}_coda_energy'], feat[f'{comp}_p_energy'])

        rows.append(feat)


df_features = pd.DataFrame(rows)
df_features.to_csv(features_path, index=False)
print('Saved:', features_path, 'shape=', df_features.shape)
df_features.head()


## Train/Evaluate EBM (only model)

We keep only the 3-segment feature table (`data_features.csv`) and only EBM.


In [ ]:
df = pd.read_csv(features_path)

y = df['label'].astype(int).values
X = df.drop(columns=['event_id', 'label', 'event_type']).copy()

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

ebm = ExplainableBoostingClassifier(random_state=0, interactions=0)
ebm.fit(X_train, y_train)

proba = ebm.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print({
    'accuracy': accuracy_score(y_test, pred),
    'f1': f1_score(y_test, pred),
    'roc_auc': roc_auc_score(y_test, proba),
})
print(classification_report(y_test, pred, target_names=['foreshock', 'aftershock']))


In [ ]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(4, 4))
plt.imshow(cm, interpolation='nearest')
plt.title('EBM confusion matrix')
plt.colorbar()
plt.xticks([0, 1], ['foreshock', 'aftershock'])
plt.yticks([0, 1], ['foreshock', 'aftershock'])
plt.xlabel('Predicted')
plt.ylabel('True')
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha='center', va='center')
plt.show()


## EBM interpretation (native)


In [ ]:
def ebm_top_importances(model, topk=20):
    ge = model.explain_global().data()
    out = pd.DataFrame({'feature': ge['names'], 'importance': ge['scores']})
    return out.sort_values('importance', ascending=False).head(topk)

top = ebm_top_importances(ebm, topk=25)
top.head(10)


In [ ]:
import re

def segment_from_feature(name):
    if '_noise_' in name:
        return 'noise'
    if '_p_' in name:
        return 'p'
    if '_coda_' in name:
        return 'coda'
    return 'cross_segment'

def channel_from_feature(name):
    m = re.match(r'^([ENZ])_', name)
    return m.group(1) if m else 'derived'

summary = top.copy()
summary['segment'] = summary['feature'].map(segment_from_feature)
summary['channel'] = summary['feature'].map(channel_from_feature)
summary.groupby(['segment', 'channel'], as_index=False)['importance'].sum().sort_values('importance', ascending=False)
